# Apple Retail Sales Analysis: Uncovering Key Insights and Strategic Opportunities

This notebook presents a comprehensive analysis of Apple retail sales data, aiming to extract meaningful insights into product performance, customer behavior, and store effectiveness. By integrating various datasets, we explore sales trends, identify top-performing products and locations, and propose data-driven strategies for enhancing sales and customer engagement.

## 1. Environment Setup and Data Loading

We begin by importing the necessary libraries for data manipulation and visualization. Following this, we load the raw datasets into Pandas DataFrames, which form the foundation for our analysis.

In [ ]:
import numpy as np  
import pandas as pd  
import matplotlib.pyplot as plt  
import os  

# List all files available in the input directory
print("Files in the input directory:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Loading Datasets

The analysis leverages five distinct datasets, each providing a piece of the overall retail sales puzzle:
- `category.csv`: Contains information about product categories.
- `products.csv`: Details individual product information.
- `sales.csv`: Records transaction-level sales data.
- `stores.csv`: Provides details about retail store locations.
- `warranty.csv`: Contains information regarding product warranties and claims.

These datasets are loaded and will be merged to create a comprehensive view of the sales ecosystem.

In [ ]:
df_category = pd.read_csv('/kaggle/input/apple-retail-sales-dataset/category.csv')
df_products = pd.read_csv('/kaggle/input/apple-retail-sales-dataset/products.csv')
df_sales =  pd.read_csv('/kaggle/input/apple-retail-sales-dataset/sales.csv')
df_stores =  pd.read_csv('/kaggle/input/apple-retail-sales-dataset/stores.csv')
df_warranty = pd.read_csv('/kaggle/input/apple-retail-sales-dataset/warranty.csv')

## 2. Initial Data Exploration and Preprocessing

Before merging the datasets, we perform an initial inspection of each DataFrame. This step helps us understand their structure, identify potential issues, and prepare them for integration.

### Inspecting `df_category`

In [ ]:
print("df_category head:")
print(df_category.head())
print("\ndf_category shape:")
print(df_category.shape)

### Inspecting `df_products`

In [ ]:
print("df_products head:")
print(df_products.head())
print("\ndf_products shape:")
print(df_products.shape)

### Inspecting `df_sales`

In [ ]:
print("df_sales head:")
print(df_sales.head())

### Inspecting `df_stores`

In [ ]:
print("df_stores head:")
print(df_stores.head())

### Inspecting `df_warranty`

In [ ]:
print("df_warranty head:")
print(df_warranty.head())

### Standardizing Column Names

To ensure consistency across DataFrames and facilitate merging, we convert all column names in `df_products` and `df_stores` to lowercase. This prevents potential issues with case-sensitive column matching.

In [ ]:
df_products.columns = df_products.columns.str.lower()
df_stores.columns = df_stores.columns.str.lower()

## 3. Data Integration: Merging DataFrames

We now systematically merge the individual DataFrames to create a single, comprehensive dataset (`df`) for holistic analysis. The merges are performed sequentially based on common identifying keys.

### Merging `df_category` and `df_products`

The `df_category` and `df_products` DataFrames are merged on `category_id` to link products with their respective categories.

In [ ]:
df = pd.merge(left=df_category, right=df_products, on='category_id')
print("Shape after merging category and products:", df.shape)
print("\nFirst 5 rows after merge:")
print(df.head())

### Merging with `df_sales`

Next, we integrate the sales transaction data by merging `df` with `df_sales` on `product_id`.

In [ ]:
df = pd.merge(left=df, right=df_sales, on='product_id')
print("Shape after merging sales:", df.shape)

### Merging with `df_stores`

Store information is added by merging `df` with `df_stores` on `store_id`.

In [ ]:
df = pd.merge(left=df, right=df_stores, on='store_id')
print("Shape after merging stores:", df.shape)

### Merging with `df_warranty`

Finally, warranty information is incorporated using a `left` merge on `sale_id`. This ensures that all sales records are retained, even if they do not have a corresponding warranty claim.

In [ ]:
df = pd.merge(left=df, right=df_warranty, on='sale_id', how='left')
print("Shape after merging warranty:", df.shape)

### Post-Merge Data Overview

After all merges are complete, we inspect the consolidated DataFrame's information (`info()`) and check for any missing values (`isnull().sum()`). This helps us understand the data types and completeness of our integrated dataset.

In [ ]:
print("DataFrame Info after all merges:")
df.info()

print("\nMissing values count after all merges:")
print(df.isnull().sum())

### Handling Missing Warranty Data

The `warranty` DataFrame was left-merged, which means sales without warranty claims will have `NaN` values in `claim_id`, `claim_date`, and `repair_status`. We impute these missing values to ensure data integrity for further analysis:
- `claim_id`: Filled with `-1` to indicate no claim.
- `claim_date`: Filled with a placeholder date '00-00-1990'.
- `repair_status`: Filled with `'unclaimed'`.

In [ ]:
df['claim_id'].fillna(-1, inplace=True)
df['claim_date'].fillna('00-00-1990', inplace=True)
df['repair_status'].fillna('unclaimed', inplace=True)

We verify the data types and null counts again to confirm the successful imputation.

In [ ]:
print("DataFrame Info after handling missing warranty data:")
df.info()
print("\nFirst 5 rows after cleaning:")
print(df.head())

## 4. Feature Engineering and Data Refinement

To enhance our analysis, we create new features and refine existing ones. This includes calculating total sales price and simplifying product names for better aggregation.

### Calculating `total_price`

A `total_price` column is calculated by multiplying `price` and `quantity` for each sale. This represents the total revenue generated from each transaction line item.

In [ ]:
df['total_price'] = df['price'] * df['quantity']

### Standardizing Product Names

The `product_name` column often contains detailed descriptions. To facilitate broader product category analysis, we extract a simplified `first_prod` name. For products starting with 'Apple', we take the first two words (e.g., 'Apple Watch', 'Apple iPhone'), otherwise, we take the first word.

In [ ]:
df['first_prod'] = df['product_name'].apply(lambda x : ' '.join(x.split(' ')[:2]) if x.startswith('Apple') else x.split(' ')[0])
print("Unique simplified product names:")
print(df['first_prod'].unique())

### Dropping Redundant ID Columns

Once the data is merged, `category_id` and `product_id` are no longer needed as individual columns for aggregated analysis and can be safely dropped to streamline the DataFrame.

In [ ]:
df.drop(columns=['category_id', 'product_id'], inplace=True)

## 5. Exploratory Data Analysis and Key Insights

This section delves into the consolidated data to uncover sales performance, product popularity, and geographic trends.

### Overall Sales Performance by Product Category

We examine the total sales revenue generated by each major product category (`first_prod`). This helps identify the top revenue drivers for Apple.

In [ ]:
# Group by simplified product name and sum total price, then convert to crores for readability
total_sales_by_product = round(df.groupby('first_prod')['total_price'].sum() / 10_000_000, 2)
print("Total Sales by Product Category (in Crores):")
print(total_sales_by_product.sort_values(ascending=False))

The analysis confirms that **iPhone**, **iPad**, and **MacBook** are significant revenue contributors, demonstrating their strong market presence and customer appeal. Apple Watch and Mac also show substantial sales figures, reinforcing the strength of Apple's core product ecosystem.

### Deep Dive into MacBook Sales by Store

Given the strong performance of MacBooks, we investigate which stores are driving the most sales for this product line. Identifying top-performing stores can inform targeted marketing and inventory strategies.

In [ ]:
macbook_sales_by_store = df[df['first_prod'] == 'MacBook'].groupby('store_name')['total_price'].sum().sort_values(ascending=False).head(8)
print("Top 8 Stores by MacBook Sales:")
print(macbook_sales_by_store)

Stores like 'Apple Store, Regent Street', 'Apple Store, Fifth Avenue', and 'Apple Store, The Grove' consistently lead in MacBook sales. These flagship locations likely benefit from high foot traffic and strong brand presence, making them crucial hubs for high-value product sales. Recognizing these top performers can guide resource allocation and strategic initiatives.

### Monthly Sales Trends for Top Stores (MacBook)

To understand the seasonality and growth patterns of MacBook sales, we visualize monthly sales for the top 9 performing stores. This provides a dynamic perspective on their sales trajectories over time.

In [ ]:
df['sale_date'] = pd.to_datetime(df['sale_date'])

top_stores = df.groupby('store_name')['total_price'].sum().nlargest(9).index

df_top_macbook_sales = df[df['store_name'].isin(top_stores) & (df['first_prod'] == 'MacBook')]

df_plot = df_top_macbook_sales.pivot_table(
    index='sale_date',
    columns='store_name',
    values='total_price',
    aggfunc='sum'
).sort_index()

df_monthly = df_plot.resample('M').sum().fillna(0)

df_rolling = df_monthly.rolling(window=2, min_periods=1).sum()

df_rolling_millions = df_rolling / 1_000_000

fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, store in enumerate(top_stores):
    if store in df_rolling_millions.columns:
        axes[i].plot(df_rolling_millions.index, df_rolling_millions[store], marker='o', markersize=3, linestyle='-', color='tab:blue')
        axes[i].set_title(store)
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].grid(True, linestyle='--', alpha=0.6)
    else:
        axes[i].set_title(f"{store} (No MacBook Sales Data)")
        axes[i].text(0.5, 0.5, 'No Data', horizontalalignment='center', verticalalignment='center', transform=axes[i].transAxes, color='gray')

plt.suptitle('Monthly MacBook Sales for Top 9 Stores (in Millions)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
plt.show()

The visualizations reveal varying sales patterns across different stores. Some stores show consistent performance, while others might have more fluctuating sales, potentially indicating localized market dynamics or promotional impacts. This granular view helps in understanding individual store performance and identifying opportunities for growth or intervention.

### Product Popularity by Quantity Sold

Beyond revenue, understanding which products are sold in the highest quantities provides insights into their sheer popularity and market penetration. This can be crucial for inventory management and marketing focus.

In [ ]:
quantity_by_product = df.groupby('first_prod')['quantity'].sum().sort_values(ascending=False).head(5)
print("Top 5 Products by Quantity Sold:")
print(quantity_by_product)

As expected, the **iPhone** leads significantly in quantity sold, highlighting its status as Apple's flagship product. **iPad**, **MacBook**, and **Apple Watch** follow, reinforcing their widespread adoption. This hierarchy of quantity sold generally aligns with revenue, suggesting a strong correlation between product popularity and financial contribution.

### Geographic Analysis: iPhone Sales by City

Understanding the geographic distribution of sales for key products like the iPhone can inform regional marketing strategies and store placement decisions. We identify the top cities for iPhone sales by quantity.

In [ ]:
df_iphone = df[df['first_prod'] == 'iPhone']
iphone_sales_by_city = df_iphone.groupby('city')['quantity'].sum().sort_values(ascending=False).head(5)
print("Top 5 Cities by iPhone Quantity Sold:")
print(iphone_sales_by_city)

Dubai, London, and New York emerge as the top cities for iPhone sales. This indicates strong market demand and customer loyalty in these metropolitan areas. Such insights are invaluable for tailoring localized campaigns and potentially expanding service offerings in these high-performing regions.

## 6. Strategic Recommendations

Based on the insights derived from the data analysis, we propose several strategic initiatives to drive further growth and enhance customer loyalty.

### Cross-Sell Marketing Strategy: iPhone Buyers to Ecosystem Expansion

**Objective:** Design a data-driven cross-sell strategy for Apple products, targeting iPhone purchasers to encourage the acquisition of additional Apple ecosystem products.

--- 

**1. Initial Trigger: iPhone Purchase**

*   When a customer purchases an **iPhone**, they become eligible for a **20% discount on any two additional Apple products** (e.g., iPad, MacBook, Apple Watch, Mac).
*   **Limitations:** This offer is valid for a **5-year period** from the iPhone purchase date and applies to a maximum of **two products per eligible purchase event**.

--- 

**2. Personalized Push Notifications**

*   **Data-Driven Targeting:** Leverage customer data, including browsing history, search patterns, app usage, and wishlist activity, to identify potential interest in other products.
*   **Optimal Timing:** Determine peak engagement times for individual users to send push notifications when they are most receptive to product interactions.

--- 

**3. Intelligent Recommendation Logic**

*   **Behavioral Signals:** Monitor repeated views of specific product categories (e.g., laptops, smartwatches) to infer high purchase intent.
*   **Problem-Solving Messaging:** Craft messages that directly address potential user pain points. For instance, if a user frequently searches for "laptop overheating issues," a notification could highlight the MacBook's superior cooling capabilities.
*   **Tailored Personalization:** Recommendations should adapt to individual user preferences, focusing on aspects like performance, battery life, design, or health tracking features.

--- 

**4. Attention-Grabbing Presentation**

*   **Compelling Hook:** Start notifications with a solution-oriented statement. Avoid generic ads in favor of messaging that highlights immediate relevance and benefit.
*   **Visual and Textual Appeal:** The initial 3–5 seconds of the notification should capture attention with concise, problem-solving visuals and text.

--- 

**5. Conversion Objective**

*   **Goal:** Guide eligible iPhone buyers towards purchasing one or both of their discounted products.
*   **Mechanism:**
    1.  Identify customers who have recently purchased an iPhone.
    2.  Continuously monitor their digital behavior for interest in other Apple products.
    3.  Deploy time-optimized, personalized push notifications highlighting problem-solving benefits.
    4.  Streamline the path to purchase with clear calls to action.

--- 

**Based on Product Quantity Data:**

| Product       | Quantity Sold (Approx.) |
|---------------|-------------------------|
| iPhone        | 834,006                 |
| iPad          | 645,303                 |
| MacBook       | 640,336                 |
| Apple Watch   | 582,445                 |
| Mac           | 385,329                 |

*   **Scenario:** A customer buys an iPhone, making them eligible for a 20% discount on any two of the remaining products.
*   **Example Push Notification:**
    > "Struggling with slow performance on your current laptop? The MacBook is engineered for speed and efficiency. As a valued iPhone owner, enjoy 20% off this and one more Apple product for the next 5 years!"

--- 

**Benefits of This Strategy:**

1.  **Ecosystem Reinforcement:** Encourages cross-selling and deepens customer engagement within the Apple ecosystem.
2.  **Loyalty and Retention:** Converts high-intent users into loyal, multi-product buyers over an extended period.
3.  **Brand Integrity:** Maintains Apple’s premium brand image by focusing on value and problem-solving rather than broad, undifferentiated discounts.
4.  **Optimized Conversion:** Maximizes conversion rates through precise timing and hyper-personalization.

This strategy leverages data-driven personalization and behavioral insights to efficiently expand customer purchases within the Apple ecosystem, balancing long-term loyalty with profitability.

### Localized Engagement Strategies for Top iPhone Markets

Given that cities like Dubai and London are significant markets for iPhones, tailored local initiatives can further strengthen brand loyalty and drive sales for other products.

*   **For Dubai:** Recognizing a potential interest in digital content and e-reading, Apple could offer a complimentary one or two-year subscription to a premium tech-based e-book or magazine app to iPhone purchasers. This aligns with lifestyle and reinforces the utility of Apple devices for content consumption.
*   **For London:** Given London's strong interest in technology and innovation, particularly robotics, Apple could implement a surprise reward program. For example, every Xth iPhone customer in top London stores could receive a mini-robot. This unique, innovative gift serves as a constant reminder of Apple's innovative spirit and can generate significant buzz and repeat purchases.

### Rewarding High-Quantity Purchases

The data shows instances of customers purchasing multiple units (e.g., 10 quantities) of a single product. Recognizing and rewarding such high-value transactions can foster deeper loyalty.

In [ ]:
max_quantity = df['quantity'].max()
print(f"Maximum quantity purchased in a single transaction: {max_quantity}")
print("\nTransactions with maximum quantity purchased:")
print(df[df['quantity'] == max_quantity])

Several transactions show a quantity of 10 units. To acknowledge and appreciate these significant purchases, Apple could introduce a 'surprise and delight' program for customers purchasing 15 or more units in a single transaction. This could range from personalized thank-you gifts to exclusive early access to new product launches, creating memorable experiences for high-volume buyers.

## 7. Warranty Claims Analysis

Understanding warranty claim patterns is crucial for product quality assessment and customer service improvements. We analyze the distribution of repair statuses across different product categories.

In [ ]:
# Filter for records where a claim was actually made (claim_id is not -1)
df_claimed = df[df['claim_id'] != -1]

# Create a crosstabulation of repair status by category name
claims_crosstab = pd.crosstab(df_claimed['repair_status'], df_claimed['category_name']).T
print("Warranty Claim Status by Product Category:")
print(claims_crosstab)

The crosstabulation provides a detailed view of warranty claims across various product categories and their repair statuses:

*   **Accessories and Smartphones** show the highest number of claims across all statuses (Completed, In Progress, Pending, Rejected), which is expected given their high sales volume and frequent usage.
*   **Laptops and Tablets** also have a significant number of claims, indicating a need to monitor quality control for these devices, especially regarding repairs in progress or pending.
*   **Subscription Services** show a notable number of 'In Progress' and 'Pending' claims, which might relate to billing issues, service disruptions, or account management, rather than physical repairs.
*   **Smart Speakers and Streaming Devices** have comparatively fewer claims, suggesting either higher reliability or lower sales volume compared to core products.

Monitoring categories with a high proportion of 'Rejected' or 'Pending' claims can highlight areas where customer communication or product diagnostics could be improved. A high volume of 'In Progress' claims might also suggest bottlenecks in the repair process that could impact customer satisfaction.

## 8. Conclusion

This analysis has provided a multi-faceted view of Apple's retail sales landscape. We've identified core revenue drivers, top-performing stores, key geographic markets, and insights into warranty claim patterns. The proposed strategies, including a data-driven cross-sell program, localized marketing initiatives, and high-quantity purchase rewards, are designed to leverage these insights to foster stronger customer relationships and drive sustained growth within the Apple ecosystem.